# Tratamento e Cruzamento de Dados — Paraíba
Censo Escolar (INEP) + Taxas de Rendimento Escolar (INEP)


In [37]:
# 1. Imports e configuração de caminhos
import pandas as pd

CAMINHO_CENSO = "data/raw/microdados_censo_escolar_2025/dados/Tabela_Escola_2025.csv"          
CAMINHO_RENDIMENTO = "data/raw/tx_rend_escolas_2025/tx_rend_escolas_2025.xlsx" 



## 1. Carregar e filtrar o Censo Escolar

In [38]:
censo = pd.read_csv(CAMINHO_CENSO, sep=";", encoding="latin-1", low_memory=False)
print(censo.shape)
censo.head()


(214192, 302)


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,IN_ESP_EXCLUSIVA_MEDIO_FIC,IN_ESP_EXCLUSIVA_MEDIO_NORMAL,IN_COMUM_EJA_FUND,IN_COMUM_EJA_MEDIO,IN_COMUM_EJA_PROF,IN_ESP_EXCLUSIVA_EJA_FUND,IN_ESP_EXCLUSIVA_EJA_MEDIO,IN_ESP_EXCLUSIVA_EJA_PROF,IN_COMUM_PROF,IN_ESP_EXCLUSIVA_PROF
0,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
# Filtrar apenas escolas da Paraíba
UF = "PB"
censo_pb = censo[censo["SG_UF"] == UF].copy()
print(censo_pb.shape)


(5641, 302)


In [ ]:
# Selecionar colunas de interesse

colunas_censo = [
    "CO_ENTIDADE", "NO_ENTIDADE", "CO_MUNICIPIO", "NO_MUNICIPIO",
    "TP_LOCALIZACAO",              #TP_LOCALIZACAO 1 = urbana, 2 = rural
    "TP_DEPENDENCIA",              #TP_DEPENDENCIA 1 = federal, 2 = estadual, 3 = municipal, 4 = privada

    # --- Infraestrutura básica (água, energia, esgoto) ---
    "IN_AGUA_INEXISTENTE",         # não há nenhuma fonte de água # água imprópria para consumo, mesmo tendo fonte
    "IN_ENERGIA_INEXISTENTE",      # não há energia elétrica
    "IN_ESGOTO_INEXISTENTE",
    "IN_BANHEIRO",

    # --- Agua e comida ---
    "IN_ALIMENTACAO",
    "IN_AGUA_POTAVEL",

    
    # --- Infraestrurua de serviços de pesquisa e tecnologia ---
    "IN_INTERNET",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_LABORATORIO_INFORMATICA",
    
         

    # --- Acessibilidade nas vias de circulação interna ---
    "IN_ACESSIBILIDADE_INEXISTENTE",  # nenhum recurso de acessibilidade listado

    # --- Climatização e ventilação ---
    "QT_SALAS_UTILIZA_CLIMATIZADAS"
]

print(censo_pb.shape)
censo_pb = censo_pb[[c for c in colunas_censo if c in censo_pb.columns]]
censo_pb.head()


(5641, 302)


,CO_ENTIDADE,NO_ENTIDADE,CO_MUNICIPIO,NO_MUNICIPIO,TP_LOCALIZACAO,TP_DEPENDENCIA,IN_AGUA_INEXISTENTE,IN_AGUA_POTAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_INEXISTENTE,IN_ACESSIBILIDADE_INEXISTENTE
58503,25033158,EMEIF MAE IAIA,2500106,Água Branca,1,3,0.0,1.0,0.0,0.0,0.0
58504,25033204,ECI JOSE NOMINANDO,2500106,Água Branca,1,2,0.0,1.0,0.0,0.0,0.0
58505,25033301,EMEF GUALTERINA ALENCAR VIDAL,2500106,Água Branca,2,3,NaN,NaN,NaN,NaN,NaN
58506,25033352,EMEF JOSE BATISTA DIAS,2500106,Água Branca,2,3,NaN,NaN,NaN,NaN,NaN
58507,25033395,EMEF MANOEL VICENTE LEITE,2500106,Água Branca,2,3,NaN,NaN,NaN,NaN,NaN


## 2. Carregar e filtrar as Taxas de Rendimento Escolar

In [41]:
rendimento = pd.read_excel(CAMINHO_RENDIMENTO, engine="openpyxl", skiprows=8)
print(rendimento.shape)
rendimento.head()


(126686, 63)


,NU_ANO_CENSO,NO_REGIAO,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,CO_ENTIDADE,NO_ENTIDADE,NO_CATEGORIA,NO_DEPENDENCIA,1_CAT_FUN,...,3_CAT_FUN_06,3_CAT_FUN_07,3_CAT_FUN_08,3_CAT_FUN_09,3_CAT_MED,3_CAT_MED_01,3_CAT_MED_02,3_CAT_MED_03,3_CAT_MED_04,3_CAT_MED_NS
0,2025,Norte,RO,1100015.0,Alta Floresta D'Oeste,11022558.0,EIEEF HAP BITT TUPARI,Rural,Estadual,100,...,--,--,--,--,--,--,--,--,--,--
1,2025,Norte,RO,1100015.0,Alta Floresta D'Oeste,11024372.0,EMEIEF ANA NERY,Rural,Municipal,98.6,...,0,0,0,0,--,--,--,--,--,--
2,2025,Norte,RO,1100015.0,Alta Floresta D'Oeste,11024666.0,EMEIEF BOA ESPERANCA,Rural,Municipal,98.4,...,0,0,0,0,--,--,--,--,--,--
3,2025,Norte,RO,1100015.0,Alta Floresta D'Oeste,11024682.0,EEEFM EURIDICE LOPES PEDROSO,Urbana,Estadual,98.9,...,0,0,0,0,0,0,0,0,--,--
4,2025,Norte,RO,1100015.0,Alta Floresta D'Oeste,11024828.0,EMEIEF IZIDORO STEDILE,Rural,Municipal,96.5,...,0,0,0,0,--,--,--,--,--,--


In [42]:
rendimento_pb = rendimento[rendimento["SG_UF"] == "PB"].copy()
print(rendimento_pb.shape)


(3782, 63)


In [43]:
# Selecionar coluna de taxa — Abandono, Anos Iniciais do Ensino Fundamental
colunas_rendimento = [
    "CO_ENTIDADE",
    "3_CAT_FUN_AI"    # Taxa de Abandono — Ensino Fundamental, Anos Iniciais
]

print(rendimento_pb.shape)
rendimento_pb = rendimento_pb[[c for c in colunas_rendimento if c in rendimento_pb.columns]]
rendimento_pb.head()


(3782, 63)


,CO_ENTIDADE,3_CAT_FUN_AI
40156,25033158.0,0
40157,25033204.0,--
40158,25033425.0,0
40159,25033476.0,0
40160,25033514.0,0


In [44]:
print(rendimento.columns.tolist())

['NU_ANO_CENSO', 'NO_REGIAO', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'CO_ENTIDADE', 'NO_ENTIDADE', 'NO_CATEGORIA', 'NO_DEPENDENCIA', '1_CAT_FUN', '1_CAT_FUN_AI', '1_CAT_FUN_AF', '1_CAT_FUN_01', '1_CAT_FUN_02', '1_CAT_FUN_03', '1_CAT_FUN_04', '1_CAT_FUN_05', '1_CAT_FUN_06', '1_CAT_FUN_07', '1_CAT_FUN_08', '1_CAT_FUN_09', '1_CAT_MED', '1_CAT_MED_01', '1_CAT_MED_02', '1_CAT_MED_03', '1_CAT_MED_04', '1_CAT_MED_NS', '2_CAT_FUN', '2_CAT_FUN_AI', '2_CAT_FUN_AF', '2_CAT_FUN_01', '2_CAT_FUN_02', '2_CAT_FUN_03', '2_CAT_FUN_04', '2_CAT_FUN_05', '2_CAT_FUN_06', '2_CAT_FUN_07', '2_CAT_FUN_08', '2_CAT_FUN_09', '2_CAT_MED', '2_CAT_MED_01', '2_CAT_MED_02', '2_CAT_MED_03', '2_CAT_MED_04', '2_CAT_MED_NS', '3_CAT_FUN', '3_CAT_FUN_AI', '3_CAT_FUN_AF', '3_CAT_FUN_01', '3_CAT_FUN_02', '3_CAT_FUN_03', '3_CAT_FUN_04', '3_CAT_FUN_05', '3_CAT_FUN_06', '3_CAT_FUN_07', '3_CAT_FUN_08', '3_CAT_FUN_09', '3_CAT_MED', '3_CAT_MED_01', '3_CAT_MED_02', '3_CAT_MED_03', '3_CAT_MED_04', '3_CAT_MED_NS']


## 3. Padronizar a chave de junção (`CO_ENTIDADE`)

In [45]:
censo_pb["CO_ENTIDADE"] = censo_pb["CO_ENTIDADE"].astype(str).str.strip()
rendimento_pb["CO_ENTIDADE"] = rendimento_pb["CO_ENTIDADE"].astype(str).str.strip()


## 4. Tratar valores ausentes / códigos especiais
O INEP usa códigos como `SIR` (sem informação de rendimento). Trate antes do merge.

In [46]:
# Ver valores únicos nas colunas de taxa para identificar códigos especiais
for col in ["TAXA_APROVACAO", "TAXA_REPROVACAO", "TAXA_ABANDONO"]:
    if col in rendimento_pb.columns:
        print(col, rendimento_pb[col].unique()[:10])


In [47]:
# Exemplo: substituir códigos especiais por NaN e converter para numérico
for col in ["TAXA_APROVACAO", "TAXA_REPROVACAO", "TAXA_ABANDONO"]:
    if col in rendimento_pb.columns:
        rendimento_pb[col] = pd.to_numeric(
            rendimento_pb[col].replace(["SIR", "--", ""], pd.NA), errors="coerce"
        )


## 5. Fazer o merge (cruzamento das bases)

In [48]:
# 1. Padronizar o tipo da coluna CO_ENTIDADE em ambas as tabelas
censo_pb["CO_ENTIDADE"] = pd.to_numeric(censo_pb["CO_ENTIDADE"], errors="coerce").astype("Int64")
rendimento_pb["CO_ENTIDADE"] = pd.to_numeric(rendimento_pb["CO_ENTIDADE"], errors="coerce").astype("Int64")

# 2. Agora sim, executar o merge
dataset = pd.merge(censo_pb, rendimento_pb, on="CO_ENTIDADE", how="inner")

print("Censo (PB):", censo_pb.shape[0])
print("Rendimento (PB):", rendimento_pb.shape[0])
print("Após o merge:", dataset.shape[0])
dataset.head()

Censo (PB): 5641
Rendimento (PB): 3782
Após o merge: 3782


,CO_ENTIDADE,NO_ENTIDADE,CO_MUNICIPIO,NO_MUNICIPIO,TP_LOCALIZACAO,TP_DEPENDENCIA,IN_AGUA_INEXISTENTE,IN_AGUA_POTAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_INEXISTENTE,IN_ACESSIBILIDADE_INEXISTENTE,3_CAT_FUN_AI
0,25033158,EMEIF MAE IAIA,2500106,Água Branca,1,3,0.0,1.0,0.0,0.0,0.0,0
1,25033204,ECI JOSE NOMINANDO,2500106,Água Branca,1,2,0.0,1.0,0.0,0.0,0.0,--
2,25033425,EMEF SEVERINO FIRMINO DE SANTANA,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0
3,25033476,EMEF ANDRE ALVES DO NASCIMENTO,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0
4,25033514,EMEF JOSE GOMES DE SALES,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0


In [49]:
dataset = pd.merge(censo_pb, rendimento_pb, on="CO_ENTIDADE", how="inner")

print("Censo (PB):", censo_pb.shape[0])
print("Rendimento (PB):", rendimento_pb.shape[0])
print("Após o merge:", dataset.shape[0])
dataset.head()


Censo (PB): 5641
Rendimento (PB): 3782
Após o merge: 3782


,CO_ENTIDADE,NO_ENTIDADE,CO_MUNICIPIO,NO_MUNICIPIO,TP_LOCALIZACAO,TP_DEPENDENCIA,IN_AGUA_INEXISTENTE,IN_AGUA_POTAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_INEXISTENTE,IN_ACESSIBILIDADE_INEXISTENTE,3_CAT_FUN_AI
0,25033158,EMEIF MAE IAIA,2500106,Água Branca,1,3,0.0,1.0,0.0,0.0,0.0,0
1,25033204,ECI JOSE NOMINANDO,2500106,Água Branca,1,2,0.0,1.0,0.0,0.0,0.0,--
2,25033425,EMEF SEVERINO FIRMINO DE SANTANA,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0
3,25033476,EMEF ANDRE ALVES DO NASCIMENTO,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0
4,25033514,EMEF JOSE GOMES DE SALES,2500106,Água Branca,2,3,0.0,1.0,0.0,0.0,0.0,0


In [50]:
# Se muitas escolas ficarem de fora, investigar com how="left"
faltantes = pd.merge(censo_pb, rendimento_pb, on="CO_ENTIDADE", how="left", indicator=True)
faltantes["_merge"].value_counts()


_merge
both          3782
left_only     1859
right_only       0
Name: count, dtype: int64

## 6. Validar e exportar o dataset tratado

In [51]:
print("Linhas:", dataset.shape[0])
print("Colunas:", dataset.shape[1])
print("\nNulos por coluna:")
print(dataset.isna().sum())


Linhas: 3782
Colunas: 12

Nulos por coluna:
CO_ENTIDADE                      0
NO_ENTIDADE                      0
CO_MUNICIPIO                     0
NO_MUNICIPIO                     0
TP_LOCALIZACAO                   0
TP_DEPENDENCIA                   0
IN_AGUA_INEXISTENTE              0
IN_AGUA_POTAVEL                  0
IN_ENERGIA_INEXISTENTE           0
IN_ESGOTO_INEXISTENTE            0
IN_ACESSIBILIDADE_INEXISTENTE    0
3_CAT_FUN_AI                     0
dtype: int64


In [52]:
import os

pasta = "data/processed"

os.makedirs(pasta, exist_ok=True)

caminho = os.path.join(pasta, "pb_evasao_infra_tratado.csv")

dataset.to_csv(caminho, index=False, encoding="utf-8")

print(f"Arquivo salvo em {caminho}")

Arquivo salvo em data/processed\pb_evasao_infra_tratado.csv
